# ETL в Greenplum — 30 заданий

Цель ETL — не просто перенести строки, а дать воспроизводимый, проверяемый и повторяемый результат. Все изменяемые объекты учебные и находятся в `m_razhin`.

## Результаты обучения

После **Распределённый ETL** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

ETL в MPP должен сохранять не только строки, но distribution, partitions, statistics и повторяемость. Staging изолирует вход, exchange/swap сокращает окно публикации.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

учебные staging/core, Metrica и справочники. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Определите batch key, full/incremental boundary, rejects, reconciliation, publish transaction и ANALYZE после изменения.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 150
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Слои

Raw сохраняет полученный контракт и технические поля загрузки. Staging приводит типы/формат и отделяет ошибки. DDS хранит согласованные сущности и историю. DM оптимизируется под потребление. Слои не должны дублировать друг друга без ответственности.

## 2. Batch и attempt

Batch описывает логическую порцию данных, attempt — конкретную попытку. Повтор после сбоя может иметь тот же batch_id и новый attempt_no. Audit хранит start/end, status, counts, watermark и error.

## 3. Атомарность

Пользователь не должен видеть половину загрузки. Транзакция объединяет target mutation и audit success. Но внешняя запись HDFS не всегда участвует в транзакции Greenplum, поэтому cross-system pipeline требует этапов и reconciliation.

## 4. Full refresh

Прост, когда target мал и допустимо полностью пересобрать. Опасный `TRUNCATE; INSERT` без транзакции создаёт окно пустых данных. Альтернативы: staging+swap/exchange или транзакционный replace.

## 5. Incremental load

Increment выбирает только новые/изменённые строки. Watermark должен задавать строгий порядок. Один timestamp недостаточен при одинаковом времени; используют `(updated_at, id)` или устойчивый offset.

## 6. Watermark lifecycle

Watermark читается до extract, новая граница вычисляется из принятого batch, но фиксируется только после успешной target. Если продвинуть раньше, retry потеряет строки. Если не обеспечить идемпотентность, retry создаст дубли.

## 7. Идемпотентность

Одинаковый вход и batch должны приводить к одинаковому target. Это достигается ключами, staging, delete+insert заданного slice, exchange, upsert pattern и audit. Надежда «DAG не запустится дважды» не является гарантией.

## 8. Late-arriving data

Событие может прийти позже watermark, но иметь старое business time. Варианты: overlap window с дедупликацией, отдельный correction feed, повтор периода или различение ingestion time/event time.

## 9. Greenplum upsert

Часто используют staging: определить новые/changed keys, удалить соответствующие target rows и вставить канонические версии в одной транзакции. Массовый pattern лучше построчных UPDATE, особенно для AO.

## 10. Hashdiff

Hash бизнес-атрибутов быстро отмечает изменение, но требует канонического порядка, формата типов и NULL marker. Hash collision теоретически возможна; критичные системы могут дополнительно сравнивать поля.

## 11. Delete+insert slice

Для факта с датой удобно переобрабатывать день/месяц: staging содержит полный slice, quality checks подтверждают его, затем target slice заменяется. Предикат DELETE и диапазон staging обязаны совпасть.

## 12. Partition exchange

Exchange — быстрый вариант замены большого slice. ETL готовит физически совместимую staging и выполняет metadata operation после checks. Бывшая partition остаётся отдельной таблицей и требует управляемой очистки.

## 13. Дедупликация

Определите business key, версию и tie-breaker. `DISTINCT` не выбирает правильную версию. Сохраняйте число входных версий и rejected conflicts для аудита.

## 14. SCD

Type 1 перезаписывает атрибуты. Type 2 закрывает текущий интервал и создаёт новую surrogate version. Инварианты SCD2: одна current row на key, интервалы не пересекаются, valid_from<valid_to.

## 15. Fact lookup

Факт связывается с dimension version, действовавшей в event time: `event_ts>=valid_from AND event_ts<valid_to`. JOIN только с current row исторически неверен.

## 16. Quality и severity

ERROR блокирует публикацию, WARN допускает загрузку с наблюдением, INFO описывает профиль. Правило хранит идентификатор, scope, measured, threshold и статус.

## 17. Reconciliation

Source rows = accepted + rejected. Stage accepted должно соответствовать target delta с учётом updates/deletes. Проверяют count, distinct keys, суммы, min/max и checksum.

## 18. Failure и retry

При ошибке сохраняют FAILED, SQLSTATE/message и этап. Target/watermark откатываются согласованно. Retry повторяет весь зависимый участок и не опирается на частично сохранённые временные данные без проверки.

## 19. Производительность

Bulk insert, совместимое распределение staging/target, AO storage и обработка partition slice уменьшают Motion и bloat. ANALYZE выполняют после значимой загрузки до потребительских запросов.

## 20. Порядок практики

Contract→batch audit→extract raw→typed staging→quality/rejects→dedup/change detection→target transaction→watermark→ANALYZE→reconciliation→publish success.

### Задание 1. `m_razhin.gpe_01_layers`

**Что сделать:** Создайте VIEW-карту raw→staging→dds→dm с гранулярностью.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Слой определяется ответственностью, не только схемой.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_01_layers.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',1);

### Задание 2. `m_razhin.gpe_02_batch_control`

**Что сделать:** Создайте таблицу batch_control со статусами и timestamps.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Одна строка на попытку загрузки.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_02_batch_control.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',2);

### Задание 3. `m_razhin.gpe_03_start_batch`

**Что сделать:** Реализуйте начало batch и сохраните audit.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Статус RUNNING до изменения target.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_03_start_batch.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',3);

### Задание 4. `m_razhin.gpe_04_raw_snapshot`

**Что сделать:** Загрузите неизменяемый raw snapshot источника.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Добавьте batch_id и load_dttm.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_04_raw_snapshot.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',4);

### Задание 5. `m_razhin.gpe_05_raw_reconcile`

**Что сделать:** Сверьте source/raw count и checksum.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Raw не должен незаметно менять значения.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_05_raw_reconcile.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',5);

### Задание 6. `m_razhin.gpe_06_staging_cast`

**Что сделать:** Создайте typed staging с нормализацией.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Ошибочные строки отделяйте.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_06_staging_cast.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',6);

### Задание 7. `m_razhin.gpe_07_quality_rules`

**Что сделать:** Создайте каталог quality rules и результаты.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Правило имеет severity.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_07_quality_rules.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',7);

### Задание 8. `m_razhin.gpe_08_rejects`

**Что сделать:** Сохраните rejected records с reason и batch_id.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Не теряйте исходный ключ.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_08_rejects.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',8);

### Задание 9. `m_razhin.gpe_09_full_refresh`

**Что сделать:** Реализуйте атомарный full refresh учебной target.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Пользователь не должен видеть пустое промежуточное состояние.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_09_full_refresh.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',9);

### Задание 10. `m_razhin.gpe_10_full_idempotent`

**Что сделать:** Докажите одинаковый результат двух full refresh.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Сравните count/checksum.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_10_full_idempotent.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',10);

## Уровень 2 — incremental, late data и reload

### Задание 11. `m_razhin.gpe_11_watermark`

**Что сделать:** Создайте watermark по event_date,event_id.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Составная позиция устраняет ничью timestamp.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_11_watermark.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',11);

### Задание 12. `m_razhin.gpe_12_increment`

**Что сделать:** Выберите строки строго после watermark.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Сравнивайте tuple или эквивалентный предикат.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_12_increment.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',12);

### Задание 13. `m_razhin.gpe_13_increment_load`

**Что сделать:** Загрузите increment в target без дублей.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Уникальность контролируется явно.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_13_increment_load.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',13);

### Задание 14. `m_razhin.gpe_14_advance_watermark`

**Что сделать:** Продвиньте watermark только после успешной target.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Ошибка не должна пропустить данные.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_14_advance_watermark.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',14);

### Задание 15. `m_razhin.gpe_15_rerun_increment`

**Что сделать:** Повторите batch и докажите отсутствие дублей.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Exactly-once effect строится поверх повторяемого процесса.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_15_rerun_increment.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',15);

### Задание 16. `m_razhin.gpe_16_late_arrival`

**Что сделать:** Обработайте late event старше watermark.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Нужен overlap window или отдельный канал.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_16_late_arrival.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',16);

### Задание 17. `m_razhin.gpe_17_upsert_pattern`

**Что сделать:** Реализуйте обновление изменившейся строки.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Greenplum-паттерн staging + delete/insert.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_17_upsert_pattern.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',17);

### Задание 18. `m_razhin.gpe_18_changed_rows`

**Что сделать:** Определите changed rows через hashdiff.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Канонизируйте NULL и типы.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_18_changed_rows.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',18);

### Задание 19. `m_razhin.gpe_19_delete_insert`

**Что сделать:** Перезагрузите один event_date через delete+insert.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Ограничьте транзакцию slice.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_19_delete_insert.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',19);

### Задание 20. `m_razhin.gpe_20_partition_reload`

**Что сделать:** Перезагрузите месяц через staging/exchange.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Сначала проверка диапазона.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_20_partition_reload.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',20);

## Уровень 3 — dedup, SCD, failure/retry и SLA

### Задание 21. `m_razhin.gpe_21_deduplicate`

**Что сделать:** Оставьте последнюю версию event_id.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Детерминированный tie-breaker.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_21_deduplicate.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',21);

### Задание 22. `m_razhin.gpe_22_delete_events`

**Что сделать:** Обработайте tombstone/delete feed.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Физическое или логическое удаление — явная стратегия.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_22_delete_events.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',22);

### Задание 23. `m_razhin.gpe_23_scd1`

**Что сделать:** Обновите учебное измерение по SCD Type 1.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

История не сохраняется.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_23_scd1.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',23);

### Задание 24. `m_razhin.gpe_24_scd2`

**Что сделать:** Создайте SCD2 с valid_from/to,current flag.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Интервалы одного ключа не пересекаются.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_24_scd2.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',24);

### Задание 25. `m_razhin.gpe_25_fact_lookup`

**Что сделать:** Свяжите факт с версией измерения по времени.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

JOIN по business key и valid interval.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_25_fact_lookup.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',25);

### Задание 26. `m_razhin.gpe_26_reconciliation`

**Что сделать:** Соберите source/stage/target/reject reconciliation.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

source=accepted+rejected и target delta.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_26_reconciliation.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',26);

### Задание 27. `m_razhin.gpe_27_failure`

**Что сделать:** Смоделируйте ошибку после staging и сохраните FAILED.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Watermark/target должны остаться согласованными.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_27_failure.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',27);

### Задание 28. `m_razhin.gpe_28_retry`

**Что сделать:** Повторите failed batch безопасно.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Новая attempt или управляемое продолжение.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_28_retry.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',28);

### Задание 29. `m_razhin.gpe_29_sla`

**Что сделать:** Рассчитайте duration, throughput и SLA status.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Технические метрики — часть ETL.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_29_sla.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',29);

### Задание 30. `m_razhin.gpe_30_pipeline`

**Что сделать:** Создайте итоговый audit pipeline по всем этапам.

Сохраняйте измеримый результат и audit. Повторный запуск задания не должен портить target.

<details><summary>Подсказка</summary>

Каждый stage имеет counts,status,started/finished.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Реализуйте m_razhin.gpe_30_pipeline.

In [ ]:
%%sql
-- Ручная проверка counts/checksum/idempotency.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('etl',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='etl' ORDER BY task_no;